# Лабораторна робота №2. Частина 2
**Завдання:** 1. Завантажити датасет `household_power_consumption.txt`.
2. Здійснити data cleaning (обробити пропущені значення, перетворити типи даних).

In [2]:
import pandas as pd
import numpy as np
import timeit
import warnings
import urllib.request
import zipfile
import os

warnings.filterwarnings('ignore')

print("Починаю підготовку даних... Це може зайняти хвилину-дві.")

data_dir = "data"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

zip_path = os.path.join(data_dir, "household_power_consumption.zip")
file_path = os.path.join(data_dir, "household_power_consumption.txt")
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"

if not os.path.exists(file_path):
    print("Завантаження архіву датасету з UCI...")
    urllib.request.urlretrieve(url, zip_path)
    print("Розпакування архіву...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)
    print("Файл успішно завантажено та розпаковано!")
else:
    print("Датасет вже знайдено локально. Пропускаю завантаження.")

print("Зчитування та очищення таблиці...")

df = pd.read_csv(file_path, sep=';', low_memory=False, na_values=['?'])

df = df.dropna()

df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time

print(f"Дані успішно очищено! Розмір таблиці: {df.shape}")
display(df.head())

Починаю підготовку даних... Це може зайняти хвилину-дві.
Завантаження архіву датасету з UCI...
Розпакування архіву...
Файл успішно завантажено та розпаковано!
Зчитування та очищення таблиці...
Дані успішно очищено! Розмір таблиці: (2049280, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


**Завдання:** Обрати всі записи, у яких загальна активна споживана потужність (`Global_active_power`) перевищує 5 кВт. Профілювання часу виконання.

In [4]:
def filter_high_power(dataframe):
    return dataframe[dataframe['Global_active_power'] > 5.0]

exec_time = timeit.timeit(lambda: filter_high_power(df), number=10) / 10
print(f"Час виконання: {exec_time:.5f} секунд")

res_task1 = filter_high_power(df)
print(f"Знайдено записів: {len(res_task1)}")
display(res_task1.head())

Час виконання: 0.00851 секунд
Знайдено записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,2006-12-16,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,2006-12-16,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


**Завдання:** Обрати всі записи, у яких сила струму (`Global_intensity`) лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник (`Sub_metering_2`) споживають більше, ніж бойлер та кондиціонер (`Sub_metering_3`).

In [5]:
def filter_intensity_and_submetering(dataframe):
    return dataframe[
        (dataframe['Global_intensity'] >= 19.0) & 
        (dataframe['Global_intensity'] <= 20.0) & 
        (dataframe['Sub_metering_2'] > dataframe['Sub_metering_3'])
    ]

exec_time = timeit.timeit(lambda: filter_intensity_and_submetering(df), number=10) / 10
print(f"Час виконання: {exec_time:.5f} секунд")

res_task2 = filter_intensity_and_submetering(df)
print(f"Знайдено записів: {len(res_task2)}")
display(res_task2.head())

Час виконання: 0.01292 секунд
Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,2006-12-16,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,2006-12-17,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,2006-12-17,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,2006-12-17,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,2006-12-17,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


**Завдання:** Обрати випадковим чином 500 000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [6]:
def random_sample_means(dataframe):
    sample_df = dataframe.sample(n=500000, replace=False)
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

exec_time = timeit.timeit(lambda: random_sample_means(df), number=10) / 10
print(f"Час виконання: {exec_time:.5f} секунд")

res_task3 = random_sample_means(df)
print("\nСередні величини груп споживання:")
print(res_task3)

Час виконання: 0.23904 секунд

Середні величини груп споживання:
Sub_metering_1    1.116854
Sub_metering_2    1.300094
Sub_metering_3    6.462422
dtype: float64


**Завдання:** Обрати ті записи, які після 18:00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання припадає на `Sub_metering_2` (група 2 є найбільшою). А потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [7]:
import datetime

def complex_evening_filter(dataframe):
    time_limit = datetime.time(18, 0, 0)
    filtered = dataframe[(dataframe['Time'] > time_limit) & (dataframe['Global_active_power'] > 6.0)]
    
    filtered = filtered[
        (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & 
        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    ]
    
    half_index = len(filtered) // 2
    first_half = filtered.iloc[:half_index]
    second_half = filtered.iloc[half_index:]
    
    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]

    return pd.concat([res_first, res_second])

exec_time = timeit.timeit(lambda: complex_evening_filter(df), number=10) / 10
print(f"Час виконання: {exec_time:.5f} секунд")

res_task4 = complex_evening_filter(df)
print(f"Знайдено записів після поділу та вибірки: {len(res_task4)}")
display(res_task4.head())

Час виконання: 0.11245 секунд
Знайдено записів після поділу та вибірки: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,2006-12-16,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,2006-12-16,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,2006-12-28,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,2006-12-28,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,2006-12-28,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


**Завдання:** Пронормувати та стандартизувати вибраний датасет. 
*(Для демонстрації оберемо лише числові колонки)*

In [8]:
def normalize_and_standardize(dataframe):
    # Вибираємо лише числові колонки для математичних операцій
    num_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']
    data_num = dataframe[num_cols]
    
    # Нормалізація: Min-Max Scaling (всі значення від 0 до 1)
    normalized = (data_num - data_num.min()) / (data_num.max() - data_num.min())
    
    # Стандартизація: Z-score (середнє = 0, стандартне відхилення = 1)
    standardized = (data_num - data_num.mean()) / data_num.std()
    
    return normalized, standardized

exec_time = timeit.timeit(lambda: normalize_and_standardize(df), number=1)
print(f"Час виконання: {exec_time:.5f} секунд\n")

df_norm, df_stand = normalize_and_standardize(df)

print("--- Нормалізовані дані (Min-Max) ---")
display(df_norm.head(3))

print("--- Стандартизовані дані (Z-score) ---")
display(df_stand.head(3))

Час виконання: 0.39414 секунд

--- Нормалізовані дані (Min-Max) ---


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,0.374796,0.300719,0.376090,0.377593
1,0.478363,0.313669,0.336995,0.473029
2,0.479631,0.358273,0.326010,0.473029


--- Стандартизовані дані (Z-score) ---


,Global_active_power,Global_reactive_power,Voltage,Global_intensity
0,2.955076,2.610720,-1.851816,3.098788
1,4.037084,2.770405,-2.225274,4.133799
2,4.050325,3.320431,-2.330213,4.133799


**Завдання:** Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів (наприклад, `Global_active_power` та `Global_intensity`).

In [10]:
def calc_correlations(dataframe):
    pearson_corr = dataframe['Global_active_power'].corr(dataframe['Global_intensity'], method='pearson')
    
    spearman_corr = dataframe['Global_active_power'].corr(dataframe['Global_intensity'], method='spearman')
    
    return pearson_corr, spearman_corr

exec_time = timeit.timeit(lambda: calc_correlations(df), number=1)
print(f"Час виконання: {exec_time:.5f} секунд\n")

pearson, spearman = calc_correlations(df)
print(f"Коефіцієнт Пірсона: {pearson:.5f}")
print(f"Коефіцієнт Спірмена: {spearman:.5f}")

Час виконання: 0.56507 секунд

Коефіцієнт Пірсона: 0.99889
Коефіцієнт Спірмена: 0.99537


**Завдання:** Провести One Hot Encoding категоріального атрибута.
*(Оскільки в датасеті немає явних категорій, ми створимо категорію "День тижня" з дати)*

In [6]:
df['Day_of_week'] = df['Date'].dt.day_name()
df_encoded = pd.get_dummies(df, columns=['Day_of_week'])

ohe_columns = [col for col in df_encoded.columns if 'Day_of_week' in col]
columns_to_display = ['Date'] + ohe_columns

print("Результат One Hot Encoding (випадкові 15 записів):")
display(df_encoded[columns_to_display].sample(15))

Результат One Hot Encoding (випадкові 15 записів):


,Date,Day_of_week_Friday,Day_of_week_Monday,Day_of_week_Saturday,Day_of_week_Sunday,Day_of_week_Thursday,Day_of_week_Tuesday,Day_of_week_Wednesday
1764816,2010-04-25,False,False,False,True,False,False,False
1543852,2009-11-22,False,False,False,True,False,False,False
712738,2008-04-24,False,False,False,False,True,False,False
182603,2007-04-22,False,False,False,True,False,False,False
1003572,2008-11-12,False,False,False,False,False,False,True
954707,2008-10-09,False,False,False,False,True,False,False
223151,2007-05-20,False,False,False,True,False,False,False
1958038,2010-09-06,False,True,False,False,False,False,False
1938369,2010-08-23,False,True,False,False,False,False,False
1752052,2010-04-16,True,False,False,False,False,False,False
